
# Step 03 Refined Keywords (Colab)

This notebook mirrors the keyword-extraction portion of `step_03_refined.py`. Mount Drive, load the `request.json` from `data/<usecase>/03_refined/remote/`, and run the extraction on Colab. When it finishes, copy the generated files plus an empty `done.marker` back into the `03_refined/outputs/` directory before rerunning the pipeline locally.


In [1]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
import json
from pathlib import Path, PureWindowsPath

# --- configuration (edit the values below if needed) ---
COMPUTER_NAME = "My Laptop"          # name shown under Drive > Othercomputers
PROJECT_SUBDIR = "gnn_project"       # synced repository folder
USECASE = "usecase_cyberspace"       # which usecase you are processing

project_root = Path(f"/content/drive/Othercomputers/{COMPUTER_NAME}/{PROJECT_SUBDIR}")
request_path = project_root / f"data/{USECASE}/03_refined/remote/request.json"
print("Project root:", project_root)
print("Request path:", request_path)
assert request_path.exists(), "request.json not found – run step_03 with refined_keywords.colab=true first."

with request_path.open() as f:
    req = json.load(f)

default_win_root = PureWindowsPath(
    r"C:\Users\paulb\Documents\CyD - nathan's project continuation\Code work\workspace\Paul work\gnn_project"
)
win_root = PureWindowsPath(req.get("project_root", default_win_root))

def win_to_colab(path_str: str) -> Path:
    rel = PureWindowsPath(path_str).relative_to(win_root)
    return project_root.joinpath(*rel.parts)

refined_papers = win_to_colab(req["refined_papers_csv"])
outputs = win_to_colab(req["outputs_dir"])
params = req["params"]
text_column = req.get("text_column", "Abstract")

outputs.mkdir(parents=True, exist_ok=True)
print("Refined papers:", refined_papers)
print("Outputs dir:", outputs)
print("sample_papers =", params.get("sample_papers"))
print("request file  =", request_path)


Project root: /content/drive/Othercomputers/My Laptop/gnn_project
Request path: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_cyberspace/03_refined/remote/request.json
Refined papers: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_cyberspace/03_refined/outputs/refined_papers.csv
Outputs dir: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_cyberspace/03_refined/outputs
sample_papers = 15000
request file  = /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_cyberspace/03_refined/remote/request.json


In [3]:
!pip install --quiet keybert sentence-transformers scikit-learn pyahocorasick


In [4]:
import os
import sys

import pandas as pd
from tqdm import tqdm

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from steps.step_02_core import (
    make_extractor,
    extract_keywords_batch,
    build_keyword_tables,
    DEFAULT_RANGE,
    DEFAULT_TOP_N,
    DEFAULT_NR_CANDIDATES,
)
from steps.step_02_keywords import _sync_refined_keyword_list, _normalize_keyword_series

os.chdir(project_root)


In [5]:
df_refined = pd.read_csv(refined_papers)
sample_df = df_refined.reset_index(drop=True)

kw_text_column = params.get('abstracts_text_col', text_column)
if kw_text_column not in sample_df.columns:
    kw_text_column = text_column

sample_limit = params.get('sample_papers')
if sample_limit and int(sample_limit) > 0 and len(sample_df) > int(sample_limit):
    sample_df = sample_df.sample(n=int(sample_limit), random_state=42).reset_index(drop=True)

raw_kw_csv = outputs / 'keywords_from_abstract.csv'
if params.get('compute', True):
    kind, extractor = make_extractor(params)
    extraction = extract_keywords_batch(
        sample_df[kw_text_column].fillna('').astype(str).tolist(),
        params,
        extractor=extractor,
        kind=kind,
        progress_desc='[Colab] extracting refined keywords',
        progress_enabled=True,
        threads=int(params.get('threads', 1)),
        return_per_doc=False,
    )
    flat_keywords = extraction.flat_keywords
    pd.Series(flat_keywords, name='raw').to_csv(raw_kw_csv, index=False)
else:
    if not raw_kw_csv.exists():
        raise FileNotFoundError(f"{raw_kw_csv} not found and compute=false")
    raw_df = pd.read_csv(raw_kw_csv)
    if 'raw' in raw_df.columns:
        flat_keywords = raw_df['raw'].astype(str).tolist()
    else:
        flat_keywords = raw_df.iloc[:, 0].astype(str).tolist()

len(flat_keywords)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: models/allenai-specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[Colab] extracting refined keywords: 100%|██████████| 15000/15000 [42:54<00:00,  5.83doc/s]


273003

In [6]:
tables = build_keyword_tables(flat_keywords, params)

df_keyword_list = pd.DataFrame({'Keyword': tables.cleaned_list})
df_keyword_list.to_csv(outputs / 'cleaned_keyword_list.csv', index=False)

refined_list_path = outputs / 'cleaned_keyword_list_refined.csv'
refined_df = _sync_refined_keyword_list(df_keyword_list, refined_list_path)
keep_mask = refined_df['removal'].astype(str).str.strip().eq('')
keep_norms = set(_normalize_keyword_series(refined_df.loc[keep_mask, 'Keyword']))
if not keep_norms:
    keep_norms = set(_normalize_keyword_series(df_keyword_list['Keyword']))

keyword_norms = _normalize_keyword_series(tables.df_keys['Keyword'])
mask_keep = keyword_norms.isin(keep_norms)
df_keys_filtered = tables.df_keys.loc[mask_keep, ['Keyword', 'Keyword_norm', 'Count']]

df_keys_filtered.to_csv(outputs / 'cleaned_keywords_to_build_graphs.csv', index=False)

matches_path = outputs / 'keyword_presence_matches.csv'
if not tables.df_hits.empty:
    tables.df_hits.to_csv(matches_path, index=False)
elif matches_path.exists():
    matches_path.unlink()

tables.presence_summary.to_csv(outputs / 'keyword_presence_summary.csv', index=False)
df_keys_filtered.to_parquet(outputs / 'keywords.parquet', index=False)

pd.DataFrame(
    {
        'n_papers': [int(len(sample_df))],
        'n_keywords_raw': [int(len(flat_keywords))],
        'n_keywords_unique': [int(df_keys_filtered.shape[0])],
        'topn': [int(params.get('keyword_topn', 0))],
        'source_file': [str(refined_papers)],
    }
).to_csv(outputs / 'keywords_summary.csv', index=False)


df_keys_filtered.head()


,Keyword,Keyword_norm,Count
0,low earth orbit,low earth orbit,974
1,earth orbit leo,earth orbit leo,790
2,global positioning,global positioning,575
3,navigation satellite systems,navigation satellite systems,552
4,satellite systems gnss,satellite systems gnss,440


In [7]:
print(f"Listing contents of the outputs directory: {outputs}")
!ls -l "{outputs}"

Listing contents of the outputs directory: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_cyberspace/03_refined/outputs
total 174768
-rw------- 1 root root   4319737 Mar  6 13:47 cleaned_keyword_list.csv
-rw------- 1 root root   4510863 Mar  6 13:47 cleaned_keyword_list_refined.csv
-rw------- 1 root root   9023187 Mar  6 13:47 cleaned_keywords_to_build_graphs.csv
-rw------- 1 root root      1198 Mar  5 15:54 collection_info.json
-rw------- 1 root root         2 Mar  5 15:55 done.marker
-rw------- 1 root root      1135 Mar  6 13:47 keyword_presence_matches.csv
-rw------- 1 root root       144 Mar  6 13:47 keyword_presence_summary.csv
-rw------- 1 root root   6037037 Mar  6 13:46 keywords_from_abstract.csv
-rw------- 1 root root   4555004 Mar  6 13:47 keywords.parquet
-rw------- 1 root root       196 Mar  6 13:47 keywords_summary.csv
-rw------- 1 root root     15398 Mar  6 09:46 old_top_keywords_counts.csv
-rw------- 1 root root  25234036 Mar  5 15:54 refined_abstracts.

In [8]:
print("Displaying the head of df_keys_filtered:")
display(df_keys_filtered.head())

Displaying the head of df_keys_filtered:


,Keyword,Keyword_norm,Count
0,low earth orbit,low earth orbit,974
1,earth orbit leo,earth orbit leo,790
2,global positioning,global positioning,575
3,navigation satellite systems,navigation satellite systems,552
4,satellite systems gnss,satellite systems gnss,440


In [9]:
print(f"Listing contents of the outputs directory on Drive: {outputs}")
!ls -l "{outputs}"

Listing contents of the outputs directory on Drive: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_cyberspace/03_refined/outputs
total 174768
-rw------- 1 root root   4319737 Mar  6 13:47 cleaned_keyword_list.csv
-rw------- 1 root root   4510863 Mar  6 13:47 cleaned_keyword_list_refined.csv
-rw------- 1 root root   9023187 Mar  6 13:47 cleaned_keywords_to_build_graphs.csv
-rw------- 1 root root      1198 Mar  5 15:54 collection_info.json
-rw------- 1 root root         2 Mar  5 15:55 done.marker
-rw------- 1 root root      1135 Mar  6 13:47 keyword_presence_matches.csv
-rw------- 1 root root       144 Mar  6 13:47 keyword_presence_summary.csv
-rw------- 1 root root   6037037 Mar  6 13:46 keywords_from_abstract.csv
-rw------- 1 root root   4555004 Mar  6 13:47 keywords.parquet
-rw------- 1 root root       196 Mar  6 13:47 keywords_summary.csv
-rw------- 1 root root     15398 Mar  6 09:46 old_top_keywords_counts.csv
-rw------- 1 root root  25234036 Mar  5 15:54 refined_a

In [10]:
(outputs / 'done.marker').write_text('ok')
print('All refined keyword artifacts written to', outputs)


All refined keyword artifacts written to /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_cyberspace/03_refined/outputs



Once Drive sync completes, rerun the local pipeline (without `--force`) so step 03 detects `done.marker`, copies the generated files, and continues with step 04.
